## 100 videos extraction from 2000

In [6]:
import os
import random
import shutil

RAW_VIDEOS_DIR = "D:/UGC_Project/raw_videos"
PILOT_DIR = "D:/UGC_Project/pilot_100"

if os.path.exists(PILOT_DIR):
    shutil.rmtree(PILOT_DIR)
os.makedirs(PILOT_DIR, exist_ok=True)

INDIAN_SOURCES = [
    os.path.join(RAW_VIDEOS_DIR, "youtube"),
    os.path.join(RAW_VIDEOS_DIR, "pixabay"),
]
GLOBAL_SOURCES = [
    "D:/UGC_Project/pixabay/pixabay_global",   # corrected — direct path, not under raw_videos
]

def get_all_videos(folders):
    videos = []
    for folder in folders:
        if os.path.exists(folder):
            for f in os.listdir(folder):
                if f.lower().endswith((".mp4", ".mov", ".webm")):
                    videos.append(os.path.join(folder, f))
    return videos

indian_videos = get_all_videos(INDIAN_SOURCES)
global_videos = get_all_videos(GLOBAL_SOURCES)

print(f"Available Indian videos: {len(indian_videos)}")
print(f"Available Global videos: {len(global_videos)}")

random.seed(42)
sample_indian = random.sample(indian_videos, min(65, len(indian_videos)))
sample_global = random.sample(global_videos, min(35, len(global_videos)))

pilot_videos = sample_indian + sample_global
print(f"\nTotal pilot sample: {len(pilot_videos)}")

for video_path in pilot_videos:
    shutil.copy(video_path, os.path.join(PILOT_DIR, os.path.basename(video_path)))

print(f"Files in pilot folder: {len(os.listdir(PILOT_DIR))}")

Available Indian videos: 510
Available Global videos: 350

Total pilot sample: 100
Files in pilot folder: 100


## Frames extraction on that 100 videos 

In [7]:
import cv2
import os
from tqdm import tqdm

PILOT_DIR = "D:/UGC_Project/pilot_100"
PILOT_FRAMES_DIR = "D:/UGC_Project/pilot_frames"
os.makedirs(PILOT_FRAMES_DIR, exist_ok=True)

TARGET_FPS = 5

def extract_frames(video_path, out_dir, fps=5):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if not video_fps or video_fps <= 0:
        video_fps = 25
    interval = max(int(video_fps // fps), 1)
    i, saved = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i % interval == 0:
            cv2.imwrite(os.path.join(out_dir, f"frame_{saved:04d}.jpg"), frame)
            saved += 1
        i += 1
    cap.release()
    return saved

video_files = [f for f in os.listdir(PILOT_DIR) if f.lower().endswith((".mp4", ".mov", ".webm"))]
print(f"Total pilot videos: {len(video_files)}")

success_count = 0
for fname in tqdm(video_files, desc="Extracting frames"):
    video_id = os.path.splitext(fname)[0]
    out_dir = os.path.join(PILOT_FRAMES_DIR, video_id)
    saved = extract_frames(os.path.join(PILOT_DIR, fname), out_dir, fps=TARGET_FPS)
    if saved > 0:
        success_count += 1

print(f"\nSuccessfully processed: {success_count}/{len(video_files)}")
total_frames = sum(len(os.listdir(os.path.join(PILOT_FRAMES_DIR, v))) for v in os.listdir(PILOT_FRAMES_DIR))
print(f"Total frames extracted: {total_frames}")

Total pilot videos: 100


Extracting frames: 100%|█████████████████████████████████████████████████████████████| 100/100 [43:15<00:00, 25.95s/it]


Successfully processed: 100/100
Total frames extracted: 13652
